# HachimiMT — Benchmark 1× T4 vs 2× T4 (Kaggle)

CTranslate2 hỗ trợ multi-GPU qua `device_index=[0,1]`. App hiện chỉ dùng 1 GPU.
Notebook này đo **dùng 2 T4 có nhanh hơn 1 T4 bao nhiêu** trên cùng văn bản, để
quyết định có đáng thêm multi-GPU vào app không.

**Cách dùng:** `Settings → Accelerator → GPU T4 x2` + `Internet → on`, rồi `Run All`.
Cell cuối in bảng — **báo lại** để chốt.

In [ ]:
# 1. Tải code + cài thư viện
import urllib.request, zipfile, os, shutil
ZIP_URL = "https://huggingface.co/spaces/ngocdang83/HachimiMT-demo/resolve/main/hachimimt-local.zip"
urllib.request.urlretrieve(ZIP_URL, "hachimimt-local.zip")
shutil.rmtree("hachimimt", ignore_errors=True)
with zipfile.ZipFile("hachimimt-local.zip") as z: z.extractall(".")
!pip install -q -r hachimimt/requirements.txt
import ctranslate2
print("CT2 cuda devices:", ctranslate2.get_cuda_device_count(), "(cần = 2 cho test này)")

In [ ]:
# 2. Chuẩn bị model + tokenizer + văn bản test (nhiều câu đa dạng)
import os, sys, time
sys.path.insert(0, os.path.abspath("hachimimt/src"))
from translator import HachimiTranslator, MODELS, Backend, ensure_model_files
from token_chunker import split_for_translation
from hardware import detect_hardware_profile

cfg = MODELS["HachimiMT-60"]
model_path = ensure_model_files(cfg, Backend.CT2)
ct2_dir = str(model_path / cfg.ct2_subdir)

# Dùng translator của app để có tokenizer + hàm token hóa đúng (chạy nhanh, chỉ để tokenize).
_app_tr = HachimiTranslator(detect_hardware_profile())
_app_tr.load("HachimiMT-60", Backend.CT2)

PARA = (
    "他抬头看向远处的山门，心中升起一股莫名的悸动。"
    "师兄说得对，我等修士当以大道为重，岂能因私情误了修行。"
    "她转身离去，没有再回头看一眼，仿佛一切已成定局。"
    "凌伊山掏出手机，查询起了最近开往雪霏市的机票。"
    "玄中门欲炼的丹药乃是凝婴丹，主材之一便是天婴果。"
    "他必须得抓紧时间了，否则一切都将来不及挽回。"
)
text = PARA * 600       # nhiều nghìn câu — đủ lớn để thấy chênh lệch GPU
chunks = split_for_translation(_app_tr._tokenizer, text, max_tokens=256, chunk_mode="sentence")
# token hóa sẵn (loại overhead tokenize khỏi phép đo GPU thuần)
tokenized = _app_tr._source_tokens_batch(chunks)
print(f"Văn bản: {len(text):,} ký tự → {len(chunks):,} chunk")

In [ ]:
# 3. Benchmark: 1 GPU [0] vs 2 GPU [0,1] — đo translate_batch thuần (token đã sẵn)
import time

def bench(device_index, label, reps=3):
    tr = ctranslate2.Translator(ct2_dir, device="cuda", device_index=device_index,
                                compute_type="int8_float16", inter_threads=len(device_index))
    # warmup
    tr.translate_batch(tokenized[:64], max_batch_size=96, batch_type="tokens", beam_size=1,
                       no_repeat_ngram_size=2, repetition_penalty=1.2)
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        tr.translate_batch(tokenized, max_batch_size=96, batch_type="tokens", beam_size=1,
                           no_repeat_ngram_size=2, repetition_penalty=1.2)
        times.append(time.perf_counter() - t0)
    times.sort()
    med = times[len(times)//2]
    cps = len(chunks) / med
    print(f"  {label:10}: {med:.2f}s · {cps:,.0f} chunk/s")
    del tr
    return cps

print(f"Dịch {len(chunks):,} chunk (token đã sẵn, beam=1, batch_type=tokens):")
c1 = bench([0],    "1× T4")
c2 = bench([0, 1], "2× T4")
print(f"\n=> 2 GPU nhanh hơn 1 GPU: {c2/c1:.2f}× "
      f"({'ĐÁNG' if c2/c1 > 1.4 else 'KHÔNG đáng'} thêm multi-GPU vào app)")
print("   (1.0× = không lợi · 2.0× = gấp đôi lý tưởng · model nhỏ thường <2×)")